In [ ]:
import gc, sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA_V2, OUTPUT
from src.benchmark import analytic_benchmark
from src.helper import _notebook_stem
from src.metrics import metrics, gain
from src.seq_tf import (
    FEATURE_SETS, TARGET, LOOKBACK,
    assign_sequences_to_splits, scale_sequences,
    compute_batch_size, hw_predict_aligned,
    build_tft, train_seq_model, save_seq_run,
)

## Configuration

In [ ]:
DATA_SET = 'chro_A'
NOTEBOOK_NAME = _notebook_stem()

SEED = 42
MAX_EPOCHS = 100
PATIENCE = 25
LR_PATIENCE = 8
LR_FACTOR = 0.3
HIDDEN_DIM = 64
N_HEADS = 4
NUM_LAYERS = 1
DROPOUT = 0.1
LR = 1e-3

## Feature Set Selection

Edit the cell below to choose which feature sets to train.  
Comment/uncomment lines, or add custom sets.

Available features: `delta`, `T`, `spy_ret`, `vix_lag`, `iv_lag`, `d_iv_lag`, `gamma`, `rho`

In [ ]:
# ── Pick which feature sets to run ──────────────────────────────────────────
# Option 1: Use all defaults (4F, 5F, 6F, 8F)
RUN_FEATURE_SETS = dict(FEATURE_SETS)

# Option 2: Keep only specific sets (uncomment to use)
# RUN_FEATURE_SETS = {k: v for k, v in FEATURE_SETS.items() if k in ('4F', '8F')}

# Option 3: Drop specific sets (uncomment to use)
# RUN_FEATURE_SETS = {k: v for k, v in FEATURE_SETS.items() if k not in ('6F',)}

# Option 4: Add a custom feature set (uncomment to use)
# RUN_FEATURE_SETS['3F'] = ['delta', 'T', 'spy_ret']
# RUN_FEATURE_SETS['custom'] = ['delta', 'T', 'spy_ret', 'gamma', 'rho']

print(f'Feature sets to run: {list(RUN_FEATURE_SETS.keys())}')
for name, cols in RUN_FEATURE_SETS.items():
    print(f'  {name}: {cols}')

## Load data

In [ ]:
df_train = pd.read_parquet(CLEAN_DATA_V2 / f'{DATA_SET}_train_v2.parquet')
df_val   = pd.read_parquet(CLEAN_DATA_V2 / f'{DATA_SET}_val_v2.parquet')
df_test  = pd.read_parquet(CLEAN_DATA_V2 / f'{DATA_SET}_test_v2.parquet')

df_full = pd.concat([df_train, df_val, df_test], ignore_index=True)

print(f'Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}')
print(f'Full data for sequences: {len(df_full):,} rows')

## Analytic benchmark

In [ ]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)
hw_coef = hw['coef']

## Train across feature sets

In [ ]:
OUTPUT_DIR = OUTPUT / NOTEBOOK_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_by_fs = {}
df_sorted_ref = None

for fs_name, feature_cols in RUN_FEATURE_SETS.items():
    print(f'\n{"=" * 60}')
    print(f'  Feature set: {fs_name}  ({len(feature_cols)} features)')
    print(f'  Features: {", ".join(feature_cols)}')
    print(f'{"=" * 60}')

    # Build sequences
    seq_data = assign_sequences_to_splits(
        df_full, df_train, df_val, df_test,
        feature_cols=feature_cols, target=TARGET, lookback=LOOKBACK,
    )
    X_tr, y_tr = seq_data['X_train'], seq_data['y_train']
    X_va, y_va = seq_data['X_val'], seq_data['y_val']
    X_te, y_te = seq_data['X_test'], seq_data['y_test']
    test_idx = seq_data['test_indices']
    df_sorted = seq_data['df_sorted']
    if df_sorted_ref is None:
        df_sorted_ref = df_sorted

    print(f'  Sequences — Train: {len(X_tr):,}  Val: {len(X_va):,}  Test: {len(X_te):,}')

    # Scale
    X_tr_sc, X_va_sc, X_te_sc, scaler = scale_sequences(X_tr, X_va, X_te)

    BATCH_SIZE = compute_batch_size(len(X_tr))
    print(f'  BATCH_SIZE={BATCH_SIZE:,}  LR={LR}')

    # Build model
    model = build_tft(
        n_features=len(feature_cols), lookback=LOOKBACK,
        hidden_dim=HIDDEN_DIM, n_heads=N_HEADS,
        num_layers=NUM_LAYERS, dropout=DROPOUT, seed=SEED,
    )
    print(f'  TFT params: {model.count_params():,}')

    # Train
    train_result = train_seq_model(
        model, X_tr_sc, y_tr, X_va_sc, y_va,
        epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
        patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR,
        desc=f'TFT {fs_name}',
    )

    # Predict & evaluate
    y_pred = train_result['model'].predict(X_te_sc, batch_size=BATCH_SIZE, verbose=0)
    _, _, hw_sse = hw_predict_aligned(hw_coef, df_sorted, test_idx)

    met = metrics(y_te, y_pred)
    g = gain(met['SSE'], hw_sse) * 100
    print(f'\n  {fs_name}: SSE={met["SSE"]:.4f}  RMSE={met["RMSE"]:.6f}  '
          f'Gain={g:+.2f}%  epochs={train_result["epochs"]}  '
          f'time={train_result["training_time"]:.1f}s')

    results_by_fs[fs_name] = {
        'model': train_result['model'],
        'y_pred': y_pred, 'y_true': y_te,
        'test_indices': test_idx,
        'history': train_result['history'],
        'epochs': train_result['epochs'],
        'training_time': train_result['training_time'],
        'scaler': scaler,
    }

    del X_tr, X_va, X_te, X_tr_sc, X_va_sc, X_te_sc
    gc.collect()

## Save results

In [ ]:
summary = save_seq_run(
    OUTPUT_DIR,
    results_by_fs=results_by_fs,
    hw_coef=hw_coef,
    df_sorted=df_sorted_ref,
)
print('\nMetrics Summary:')
summary

## Summary

In [ ]:
total_time = sum(r['training_time'] for r in results_by_fs.values())
print(f'\nTFT on {DATA_SET} — total training time: {total_time / 60:.1f} min')
for fs_name, res in results_by_fs.items():
    met = metrics(res['y_true'], res['y_pred'])
    _, _, hw_sse = hw_predict_aligned(hw_coef, df_sorted_ref, res['test_indices'])
    g = gain(met['SSE'], hw_sse) * 100
    print(f'  {fs_name}: SSE={met["SSE"]:.4f}  Gain={g:+.2f}%  epochs={res["epochs"]}')

## Cleanup

In [ ]:
del results_by_fs, df_full, df_sorted_ref
gc.collect()
print(f'Total training time: {total_time / 60:.1f} min')